# Differential Equations — Session 39
## Section 8.4: Matrix Exponential

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Students should be able to:

1. define $e^{At}$ by a power series;
2. differentiate the matrix exponential;
3. show that $e^{At}$ is a fundamental matrix;
4. use $e^{At}$ to solve homogeneous and nonhomogeneous systems;
5. compute $e^{At}$ for diagonal and diagonalizable matrices;
6. compute a Jordan-block exponential;
7. state the semigroup and inverse properties;
8. connect $e^{At}$ with the Laplace resolvent $(sI-A)^{-1}$.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |
|---:|---|
| 0–18 min | Definition and series |
| 18–34 min | Derivative and fundamental matrix |
| 34–52 min | Diagonalization |
| 52–68 min | Jordan blocks |
| 68–80 min | Flow and semigroup properties |
| 80–88 min | Nonhomogeneous formula and resolvent |
| 88–90 min | Exit check |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad_vec
from scipy.linalg import expm, eig
from IPython.display import display, Markdown

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=6, suppress=True)

def solve_linear_system(A, x0, t_span=(0, 10), points=1000, forcing=None):
    A = np.asarray(A, dtype=float)
    x0 = np.asarray(x0, dtype=float)
    t_eval = np.linspace(t_span[0], t_span[1], points)
    if forcing is None:
        def rhs(t, x): return A @ x
    else:
        def rhs(t, x): return A @ x + np.asarray(forcing(t), dtype=float)
    return solve_ivp(rhs, t_span, x0, t_eval=t_eval, rtol=1e-9, atol=1e-11)

print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Definition 8.4-A — Matrix exponential

For a square matrix $A$,

$$
e^{At}
=
I+At+\frac{A^2t^2}{2!}
+\frac{A^3t^3}{3!}+\cdots.
$$

### Theorem 8.4-B — Derivative

$$
\frac{d}{dt}e^{At}
=
Ae^{At}
=
e^{At}A.
$$

### Corollary 8.4-C — Fundamental matrix

$e^{At}$ is the unique matrix solution of

$$
\Phi'=A\Phi,
\qquad
\Phi(0)=I.
$$

Therefore it is a fundamental matrix.

### Theorem 8.4-D — Homogeneous IVP

$$
\mathbf X'=A\mathbf X,
\qquad
\mathbf X(0)=\mathbf X_0
$$

has solution

$$
\mathbf X(t)=e^{At}\mathbf X_0.
$$

### Theorem 8.4-E — Diagonalizable matrix

If $A=PDP^{-1}$, then

$$
e^{At}=Pe^{Dt}P^{-1}.
$$

### Theorem 8.4-F — Semigroup and inverse properties

$$
e^{A(t+s)}=e^{At}e^{As},
$$

$$
(e^{At})^{-1}=e^{-At}.
$$

### Theorem 8.4-G — Nonhomogeneous IVP

$$
\mathbf X(t)
=
e^{At}\mathbf X_0
+
\int_0^t e^{A(t-\tau)}\mathbf F(\tau)\,d\tau.
$$

### Theorem 8.4-H — Laplace-transform identity

$$
\mathcal L\{e^{At}\}
=
(sI-A)^{-1}
$$

for $s$ to the right of all eigenvalue real parts.

### Classroom Checkpoint — Jordan Exponential

If $A=\lambda I+N$ and $N^2=0$, simplify $e^{At}$.

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Matrix-series convergence

The series definition mirrors the scalar exponential. For a fixed matrix, partial sums converge rapidly for moderate $t$.

In [ ]:
def matrix_exp_partial(A, t, N):
    A = np.asarray(A, dtype=float)
    total = np.eye(A.shape[0])
    term = np.eye(A.shape[0])
    for n in range(1, N+1):
        term = term @ (A*t)/n
        total = total + term
    return total

A = np.array([[0,-1],[1,0]], float)
for N in [1,2,4,8,16]:
    approx = matrix_exp_partial(A, 2.0, N)
    error = np.linalg.norm(approx-expm(A*2.0))
    print("N =", N, "error =", error)

In [ ]:
def series_convergence_explorer(t=2.0, N=6):
    A = np.array([[0,-1],[1,0]], float)
    approx = matrix_exp_partial(A, t, N)
    exact = expm(A*t)
    print("partial sum:")
    print(approx)
    print("exact:")
    print(exact)
    print("matrix norm error:", np.linalg.norm(approx-exact))

if WIDGETS_AVAILABLE:
    interact(
        series_convergence_explorer,
        t=FloatSlider(min=-6, max=6, step=0.25, value=2),
        N=IntSlider(min=1, max=30, step=1, value=6)
    )
else:
    series_convergence_explorer()

## 2. Rotation matrix from an exponential

For

$$
A=
\begin{pmatrix}
0&-\omega\\
\omega&0
\end{pmatrix},
$$

$$
e^{At}
=
\begin{pmatrix}
\cos\omega t&-\sin\omega t\\
\sin\omega t&\cos\omega t
\end{pmatrix}.
$$

In [ ]:
omega = 1.5
A = np.array([[0,-omega],[omega,0]], float)
for time in [0, 0.5, 1.0]:
    print("t =", time)
    print(expm(A*time))

## 3. Flow map of a set

The matrix $e^{At}$ maps every initial state to its state at time $t$.

In [ ]:
def flow_map_explorer(time=1.0, case="rotation"):
    if case == "rotation":
        A = np.array([[0,-1],[1,0]], float)
    elif case == "saddle":
        A = np.array([[1,0],[0,-1]], float)
    else:
        A = np.array([[-1,2],[0,-1]], float)

    theta = np.linspace(0, 2*np.pi, 500)
    circle = np.vstack([np.cos(theta), np.sin(theta)])
    image = expm(A*time) @ circle

    plt.plot(circle[0], circle[1], label="initial unit circle")
    plt.plot(image[0], image[1], label=fr"image under $e^{{At}}$, $t={time:.2f}$")
    plt.axis("equal")
    plt.legend()
    plt.title("The matrix exponential as a flow map")
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        flow_map_explorer,
        time=FloatSlider(min=-3, max=3, step=0.1, value=1),
        case=Dropdown(options=["rotation","saddle","Jordan"], value="rotation")
    )
else:
    flow_map_explorer()

## 4. Diagonalization

If

$$
A=PDP^{-1},
$$

then powers of $A$ reduce to powers of the diagonal matrix, giving

$$
e^{At}=Pe^{Dt}P^{-1}.
$$

In [ ]:
A = np.array([[2,1],[1,2]], float)
vals, P = np.linalg.eig(A)
Dexp = np.diag(np.exp(vals*0.7))
via_diag = P @ Dexp @ np.linalg.inv(P)
direct = expm(A*0.7)

print("via diagonalization:")
print(via_diag)
print("direct expm:")
print(direct)
print("difference norm:", np.linalg.norm(via_diag-direct))

## 5. Jordan block

For

$$
A=\lambda I+N,
\qquad
N^2=0,
$$

$$
e^{At}
=
e^{\lambda t}(I+tN).
$$

The polynomial factor is the matrix-exponential origin of generalized-eigenvector solutions.

In [ ]:
lambda_value = -1.0
A = np.array([[lambda_value,1],[0,lambda_value]], float)
t = np.linspace(0, 8, 700)

entries = np.array([expm(A*ti) for ti in t])
plt.plot(t, entries[:,0,0], label="(1,1)")
plt.plot(t, entries[:,0,1], label="(1,2)")
plt.plot(t, entries[:,1,1], label="(2,2)")
plt.legend()
plt.title("Entries of a Jordan-block exponential")
plt.show()

## 6. Semigroup property

Evolving for time $t$ and then time $s$ is equivalent to evolving once for time $t+s$.

In [ ]:
A = np.array([[-1,2],[-3,-1]], float)
t1, t2 = 0.7, 1.3
left = expm(A*(t1+t2))
right = expm(A*t1) @ expm(A*t2)

print("semigroup error:", np.linalg.norm(left-right))
print("inverse error:", np.linalg.norm(np.linalg.inv(expm(A*t1))-expm(-A*t1)))

## 7. Derivative verification

A finite-difference derivative of $e^{At}$ should agree with $Ae^{At}$.

In [ ]:
A = np.array([[0,1],[-4,-0.5]], float)
t0, h = 1.2, 1e-6
finite_difference = (expm(A*(t0+h))-expm(A*(t0-h)))/(2*h)
analytic = A @ expm(A*t0)
print("derivative error:", np.linalg.norm(finite_difference-analytic))

## 8. Forced response

The same matrix exponential propagates both the initial state and each past input.

In [ ]:
A = np.array([[-1,1],[-2,-1]], float)
x0 = np.array([1,0], float)

def F(t):
    return np.array([0, np.exp(-0.2*t)])

grid = np.linspace(0, 10, 250)
formula = np.array([
    expm(A*t) @ x0 +
    quad_vec(lambda tau: expm(A*(t-tau)) @ F(tau), 0, t)[0]
    for t in grid
])

plt.plot(grid, formula[:,0], label="x")
plt.plot(grid, formula[:,1], label="y")
plt.legend()
plt.title("Nonhomogeneous response from the matrix exponential")
plt.show()

## 9. Resolvent connection

Taking the Laplace transform of

$$
\Phi'=A\Phi,
\qquad
\Phi(0)=I
$$

gives

$$
(sI-A)\mathcal L\{\Phi\}=I,
$$

so

$$
\mathcal L\{e^{At}\}=(sI-A)^{-1}.
$$

In [ ]:
s = sp.symbols("s")
A_sym = sp.Matrix([[0,-1],[1,0]])
resolvent = (s*sp.eye(2)-A_sym).inv()
display(resolvent)

## Classroom Checkpoint — Exit Check

For

$$
A=
\begin{pmatrix}
2&0\\
0&-1
\end{pmatrix},
$$

find $e^{At}$.

> Pause here. Let students commit to an answer before running the next cell.